# **[W2] Pilot Task**



## **데이터 기본 정보**

**<데이터 개요>**
- 사용 데이터: **Kaggle: Predict Customer Churn**

- URL: https://www.kaggle.com/competitions/playground-series-s6e3/overview

- 목표: 고객 이탈 가능성 예측

- 평가 지표: ROC

---

**<데이터 구성>**
- train.csv: 학습용 데이터, 594,194행
- test.csv: 평가용 데이터, 254,651행
- sample_submission.csv: 제출 예시.(본 분석에서는 필요하지 않음)

In [26]:
# 기본 설정
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

In [27]:
# Kaggle setting
from google.colab import userdata
import os
import kagglehub

os.environ["KAGGLE_USERNAME"] = userdata.get('KAGGLE_USERNAME')
os.environ["KAGGLE_KEY"] = userdata.get('KAGGLE_KEY')

path = kagglehub.competition_download('playground-series-s6e3')

print("Path to dataset files", path)

print("Files in path:", os.listdir(path))


Path to dataset files /root/.cache/kagglehub/competitions/playground-series-s6e3
Files in path: ['test.csv', 'train.csv', 'sample_submission.csv']


In [28]:
# (기본) 데이터 불러오기
train = pd.read_csv(path + '/train.csv')
test = pd.read_csv(path + '/test.csv')
submission = pd.read_csv(path + '/sample_submission.csv')

In [29]:
# 복사본 생성
train_df = train.copy()
test_df = test.copy()

In [30]:
# 데이터 형태
print(f"{train.info()}\n")

print(f"{train.columns}")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 594194 entries, 0 to 594193
Data columns (total 21 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   id                594194 non-null  int64  
 1   gender            594194 non-null  object 
 2   SeniorCitizen     594194 non-null  int64  
 3   Partner           594194 non-null  object 
 4   Dependents        594194 non-null  object 
 5   tenure            594194 non-null  int64  
 6   PhoneService      594194 non-null  object 
 7   MultipleLines     594194 non-null  object 
 8   InternetService   594194 non-null  object 
 9   OnlineSecurity    594194 non-null  object 
 10  OnlineBackup      594194 non-null  object 
 11  DeviceProtection  594194 non-null  object 
 12  TechSupport       594194 non-null  object 
 13  StreamingTV       594194 non-null  object 
 14  StreamingMovies   594194 non-null  object 
 15  Contract          594194 non-null  object 
 16  PaperlessBilling  59

In [31]:
# 데이터 형태_2
train.head(10)

,id,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,0,Male,0,Yes,Yes,29,Yes,No,DSL,Yes,...,Yes,Yes,No,No,One year,Yes,Mailed check,60.10,1653.85,No
1,1,Male,0,Yes,Yes,58,Yes,No,DSL,Yes,...,No,Yes,Yes,No,Two year,No,Credit card (automatic),69.50,3778.20,No
2,2,Male,0,Yes,No,58,Yes,Yes,Fiber optic,No,...,No,No,Yes,Yes,Month-to-month,Yes,Electronic check,100.40,5841.35,No
3,3,Female,0,No,No,1,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,69.70,70.70,Yes
4,4,Female,0,No,No,1,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.45,70.45,Yes
5,5,Male,0,Yes,Yes,1,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Month-to-month,Yes,Bank transfer (automatic),20.20,20.20,No
6,6,Female,0,Yes,Yes,24,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Month-to-month,No,Mailed check,20.40,533.60,No
7,7,Male,0,Yes,No,72,Yes,Yes,DSL,Yes,...,Yes,Yes,Yes,Yes,Two year,No,Electronic check,92.00,6827.50,No
8,8,Female,1,No,No,1,Yes,Yes,Fiber optic,No,...,Yes,No,No,No,Month-to-month,Yes,Electronic check,79.55,79.55,Yes
9,9,Male,0,No,No,55,Yes,No,Fiber optic,Yes,...,No,No,Yes,Yes,Month-to-month,Yes,Electronic check,100.05,4738.30,No


## **(TASK1-2) 데이터 전처리 및 피처 엔지니어링**

1. **이상치/결측치 처리**: 이상치/결측치가 존재하지 않으므로 해당 단계 미진행

2. **데이터 전처리**
    1) **컬럼 제거**: 모델 학습에 필요하지 않은 `id` 컬럼 제거
    2) **데이터 변환**:
        - 모델 예측에 필요한 `target data`(`Churn: 고객 이탈 여부`) 분리
        - `binary data` 자료 형태 변환(object > numberic)
    3) **피처 엔지니어링**
        > 1. Feature 1(`ServiceCount`): 서비스 이용 개수  
        > 2. Feature 2(`AvgMonthlyFee`): 월 평균 청구 금액
        
        <사전 EDA 결과 및  생성 논리>
        1. 여러 서비스를 동시에 이용할 수록 다른 상품으로의 이탈이 어려워짐 > 장기 이용 고객일수록 이탈이 줄어듦

        2. 월 요금 액수가 커질수록 이탈자가 늘어나는 경향을 보임 > 월 평균 지출액을 산출하여 이를 검증( `TotalCharges(청구된 총 금액) / tenure(서비스 이용 기간)`) 월 요금 따라 이탈자와 유지자의 차이가 심함
    4) **데이터 인코딩**
        - 범주형 데이터에 대해 `One-hot Encoding`을 진행함
        - 다른 Encoding 방법은 추후 상황에 따라 적용
    5) **데이터 스케일링**
        - 대부분의 수치형 데이터는 요금, 기간을 의미하는 데이터이므로 `StandardScaler`를 활용하여 데이터 스케일링을 진행



In [32]:
# 1. 불필요한 컬럼 제거
test_id = test['id'] # submission 제출 시 필요한 id 컬럼 별도 보관

train_proc = train.drop('id', axis=1) # 모델 훈련 시 불필요한 컬럼 제거
test_proc = test.drop('id', axis=1) # 모델 훈련 시 불필요한 컬럼 제거

print(train_proc.shape)
print(test_proc.shape)



(594194, 20)
(254655, 19)


In [33]:
# 2. target data 변환 + binary data 변환
train_proc['Churn'] = train_proc['Churn'].map({'Yes':1, 'No':0}) # Target Data

binary_cols = ['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling'] # Binary Data Columns
for col in binary_cols:
    train_proc[col] = train_proc[col].map({'Yes':1, 'No':0})
    test_proc[col] = test_proc[col].map({'Yes':1, 'No':0})

# 서비스 가입 여부에 대한 Binary Data는 이후 Feature Engineering에서 별도 처리

In [34]:
# 3. Feature Engineering
service_cols = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']

# Feature 1: 이용 서비스 개수 합계
train_proc['ServiceCount'] = train[service_cols].apply(lambda x: (x == 'Yes').sum(), axis=1)
test_proc['ServiceCount'] = test[service_cols].apply(lambda x: (x == 'Yes').sum(), axis=1)

# Feature 2: 가입 기간 대비 지출액
train_proc['AvgMonthlyFee'] = train_proc['TotalCharges'] / (train_proc['tenure']+1)
test_proc['AvgMonthlyFee'] = test_proc['TotalCharges'] / (test_proc['tenure']+1)

In [35]:
# 4. Categorical Data One-Hot Encoding
cat_cols = ['gender', 'MultipleLines', 'InternetService', 'OnlineSecurity',
            'OnlineBackup', 'DeviceProtection', 'TechSupport',
            'StreamingTV', 'StreamingMovies', 'Contract', 'PaymentMethod']

train_proc = pd.get_dummies(train_proc, columns=cat_cols, drop_first=True, dtype=int)
test_proc = pd.get_dummies(test_proc, columns=cat_cols, drop_first=True, dtype=int)

In [36]:
# 5. Data Scaling
from sklearn.preprocessing import StandardScaler

ss=StandardScaler()

num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges', 'ServiceCount', 'AvgMonthlyFee']

train_proc[num_cols] = ss.fit_transform(train_proc[num_cols])
test_proc[num_cols] = ss.transform(test_proc[num_cols])

In [37]:
# Result
train_proc.head()

,SeniorCitizen,Partner,Dependents,tenure,PhoneService,PaperlessBilling,MonthlyCharges,TotalCharges,Churn,ServiceCount,...,TechSupport_Yes,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,1,1,-0.302342,1,1,-0.185604,-0.357076,0,0.480853,...,1,0,0,0,0,1,0,0,0,1
1,0,1,1,0.854793,1,0,0.116964,0.545399,0,0.988816,...,1,0,1,0,0,0,1,1,0,0
2,0,1,0,0.854793,1,1,1.111575,1.421875,0,0.480853,...,0,0,1,0,1,0,0,0,1,0
3,0,0,0,-1.419575,1,1,0.123402,-1.029637,1,-1.043033,...,0,0,0,0,0,0,0,0,1,0
4,0,0,0,-1.419575,1,1,0.147543,-1.029743,1,-1.043033,...,0,0,0,0,0,0,0,0,1,0


## **W3**

In [38]:
# 데이터 내보내기

train_proc.to_csv('train_proc.csv', index=False)
test_proc.to_csv('test_proc.csv', index=False)

test_id.to_csv('test_id.csv', index=False)

In [40]:
# 데이터 내보내기
from google.colab import files

files.download('train_proc.csv')
files.download('test_proc.csv')
files.download('test_id.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# 데이터 불러오기(W3)

# train_W2 = pd.read_csv('train_proc.csv')
# test_W2 = pd.read_csv('test_proc.csv')
# test_id_W2 = pd.read_csv('test_id.csv')